# ASL (English) Sign Classifier — MobileNetV3-Small

Trains a 29-class image classifier (A–Z + SPACE + DEL + NOTHING) using MobileNetV3-Small
with transfer learning from ImageNet.

**Checkpoint strategy**
- Every epoch saves its own numbered `.pt` file to Drive (never overwritten).
- A `manifest.json` records val accuracy for every checkpoint so you can compare.
- A *Load & Test Any Checkpoint* cell (Step 7) lets you pick any epoch, run it on 10 val
  images, and download it if you like it.
- Resuming: re-running Step 5 automatically picks up from the last saved epoch.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**
2. Have your `kaggle.json` ready (kaggle.com → Settings → API → Create New Token)

## Step 1: Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR    = '/content/drive/MyDrive/ArSL_Project'
CROPPED_DIR    = os.path.join(PROJECT_DIR, 'asl_cropped')          # cached MediaPipe crops
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints_eng_cls')  # one .pt per epoch
RAW_DIR        = '/content/asl_raw'                                 # Colab-local (fast I/O)

for d in [PROJECT_DIR, CROPPED_DIR, CHECKPOINT_DIR, RAW_DIR]:
    os.makedirs(d, exist_ok=True)

# Do NOT pin mediapipe here — Colab already has it pre-installed, and installing
# mediapipe==0.10.14 downgrades protobuf to 4.x which breaks Colab's TensorFlow.
!pip install kaggle -q

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

try:
    import mediapipe as mp
    _ = mp.solutions.hands   # verify solutions API is present
    HAS_MEDIAPIPE = True
    print(f'MediaPipe: {mp.__version__} (solutions API OK)')
except Exception as e:
    HAS_MEDIAPIPE = False
    print(f'MediaPipe unavailable ({e})')
    print('  → crop step will fall back to center-square crop (slightly worse alignment)')

print('Setup complete.')

## Step 2: Download ASL Dataset from Kaggle
Upload your `kaggle.json` when prompted. Skipped automatically if already downloaded.

In [ ]:
EXTRACTED_DIR = os.path.join(RAW_DIR, 'asl_alphabet_train', 'asl_alphabet_train')

if os.path.isdir(EXTRACTED_DIR) and len(os.listdir(EXTRACTED_DIR)) >= 29:
    print('Dataset already on Colab disk — skipping download.')
    print(f'Folders ({len(os.listdir(EXTRACTED_DIR))}): {sorted(os.listdir(EXTRACTED_DIR))}')
else:
    from google.colab import files
    print('Upload your kaggle.json:')
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    !kaggle datasets download -d grassknoted/asl-alphabet -p {RAW_DIR} --unzip
    print(f'Done. Folders: {sorted(os.listdir(EXTRACTED_DIR))}')

## Step 3: Build MediaPipe-Cropped Dataset (cached to Drive)

Each image is cropped to the hand bounding box **exactly like inference will**.  
This closes the train/test gap that broke the old YOLO model.

- First run: ~20 min on T4. Result cached to Drive.
- Every subsequent run (or after a runtime restart): **skipped instantly**.

In [ ]:
import cv2, numpy as np, random

CLASS_NAMES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','SPACE','DEL','NOTHING'
]

# Kaggle asl-alphabet dataset: letter folders are UPPERCASE (A, B, ... Z),
# special class folders are lowercase (space, del, nothing).
FOLDER_MAP = {
    'A':'A','B':'B','C':'C','D':'D','E':'E','F':'F','G':'G','H':'H','I':'I','J':'J',
    'K':'K','L':'L','M':'M','N':'N','O':'O','P':'P','Q':'Q','R':'R','S':'S','T':'T',
    'U':'U','V':'V','W':'W','X':'X','Y':'Y','Z':'Z',
    'space':'SPACE','del':'DEL','nothing':'NOTHING'
}

# ── Crop helpers ─────────────────────────────────────────────────────────────

def _to_square_rgb(img_rgb):
    h, w = img_rgb.shape[:2]
    side = max(h, w)
    out  = np.zeros((side, side, 3), dtype=np.uint8)
    oh, ow = (side - h) // 2, (side - w) // 2
    out[oh:oh+h, ow:ow+w] = img_rgb
    return out

def _center_square_crop(img_bgr):
    """Fallback: take the center square of the image."""
    h, w = img_bgr.shape[:2]
    s    = min(h, w)
    y0, x0 = (h - s) // 2, (w - s) // 2
    return cv2.cvtColor(img_bgr[y0:y0+s, x0:x0+s], cv2.COLOR_BGR2RGB)

def _mediapipe_crop(img_bgr, hands_ctx):
    """MediaPipe hand-bbox crop → square RGB. Returns None if no hand found."""
    h, w   = img_bgr.shape[:2]
    rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    result = hands_ctx.process(rgb)
    if not result.multi_hand_landmarks:
        return None
    lms = result.multi_hand_landmarks[0]
    xs  = [lm.x for lm in lms.landmark]
    ys  = [lm.y for lm in lms.landmark]
    bw, bh = max(xs) - min(xs), max(ys) - min(ys)
    x0 = max(0, int((min(xs) - bw*0.2) * w))
    y0 = max(0, int((min(ys) - bh*0.2) * h))
    x1 = min(w, int((max(xs) + bw*0.2) * w))
    y1 = min(h, int((max(ys) + bh*0.2) * h))
    crop = rgb[y0:y1, x0:x1]
    if crop.size == 0:
        return None
    return _to_square_rgb(crop)

# ── Build cropped dataset ────────────────────────────────────────────────────

# Check if already done (all 29 classes present with >1000 train images)
done = [c for c in CLASS_NAMES
        if os.path.isdir(os.path.join(CROPPED_DIR, 'train', c))
        and len(os.listdir(os.path.join(CROPPED_DIR, 'train', c))) > 1000]

if len(done) == len(CLASS_NAMES):
    print(f'All {len(CLASS_NAMES)} classes already cropped on Drive — skipping.')
else:
    mode = 'MediaPipe hand crop' if HAS_MEDIAPIPE else 'center-square fallback'
    print(f'Cropping {len(CLASS_NAMES) - len(done)} remaining classes ({mode})...')
    random.seed(42)
    total_ok = total_skip = 0

    mp_hands_mod = mp.solutions.hands if HAS_MEDIAPIPE else None

    for folder_name, class_name in FOLDER_MAP.items():
        src = os.path.join(EXTRACTED_DIR, folder_name)
        if not os.path.isdir(src):
            print(f'  WARNING: folder not found: {src}'); continue

        train_out = os.path.join(CROPPED_DIR, 'train', class_name)
        if os.path.isdir(train_out) and len(os.listdir(train_out)) > 1000:
            print(f'  {class_name}: already done'); continue

        for split in ['train', 'val', 'test']:
            os.makedirs(os.path.join(CROPPED_DIR, split, class_name), exist_ok=True)

        imgs = sorted(f for f in os.listdir(src)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        random.shuffle(imgs)
        n = len(imgs)
        splits = {
            'train': imgs[:int(n*0.8)],
            'val':   imgs[int(n*0.8):int(n*0.9)],
            'test':  imgs[int(n*0.9):],
        }
        cls_ok = cls_skip = 0

        ctx_manager = (mp_hands_mod.Hands(static_image_mode=True, max_num_hands=1,
                                          min_detection_confidence=0.3)
                       if HAS_MEDIAPIPE else None)
        try:
            for split, img_list in splits.items():
                out_dir = os.path.join(CROPPED_DIR, split, class_name)
                for fname in img_list:
                    img = cv2.imread(os.path.join(src, fname))
                    if img is None:
                        continue
                    if HAS_MEDIAPIPE and class_name != 'NOTHING':
                        cropped = _mediapipe_crop(img, ctx_manager)
                        if cropped is None:
                            cls_skip += 1
                            continue
                    else:
                        cropped = _center_square_crop(img)
                    cv2.imwrite(os.path.join(out_dir, fname),
                                cv2.cvtColor(cropped, cv2.COLOR_RGB2BGR))
                    cls_ok += 1
        finally:
            if ctx_manager is not None:
                ctx_manager.close()

        total_ok   += cls_ok
        total_skip += cls_skip
        print(f'  {class_name}: {cls_ok} saved, {cls_skip} skipped')

    print(f'\nTotal: {total_ok} saved, {total_skip} skipped')

for split in ['train', 'val', 'test']:
    n = sum(len(os.listdir(os.path.join(CROPPED_DIR, split, c)))
            for c in CLASS_NAMES
            if os.path.isdir(os.path.join(CROPPED_DIR, split, c)))
    print(f'{split}: {n} images')

## Step 4: Build Model & Dataloaders

In [ ]:
import torch, torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

CLASS_NAMES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','SPACE','DEL','NOTHING'
]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}, classes: {NUM_CLASSES}')

# Strong augmentation — NO horizontal flip (left/right hand distinguishes letters in ASL)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.RandomAffine(degrees=15, translate=(0.1,0.1), scale=(0.85,1.15), shear=8),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = datasets.ImageFolder(os.path.join(CROPPED_DIR,'train'), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(CROPPED_DIR,'val'),   transform=val_tf)
test_ds  = datasets.ImageFolder(os.path.join(CROPPED_DIR,'test'),  transform=val_tf)

# ImageFolder sorts folders alphabetically; remap to our CLASS_NAMES order
folder_idx = train_ds.class_to_idx   # {'A':0, 'B':1, ...} in alpha order
remap = {folder_idx[c]: i for i, c in enumerate(CLASS_NAMES) if c in folder_idx}
for ds in [train_ds, val_ds, test_ds]:
    ds.targets = [remap.get(t, t) for t in ds.targets]
    ds.samples = [(p, remap.get(t, t)) for p, t in ds.samples]

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)
print(f'Head: Linear({in_features} → {NUM_CLASSES})')

## Step 5: Train — Every Epoch Saved as a Checkpoint

**Stage A** (3 epochs): backbone frozen, head only, LR 1e-3  
**Stage B** (up to 15 epochs, early-stop patience=5): last block + head, LR 1e-4 cosine  

Each epoch writes `epoch_NN.pt` to Drive and updates `manifest.json` with val accuracy.  
Re-running this cell resumes from the last completed epoch automatically.

In [ ]:
import json, time

MANIFEST_PATH = os.path.join(CHECKPOINT_DIR, 'manifest.json')

def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH) as f:
            return json.load(f)
    return {'checkpoints': [], 'best_epoch': None, 'best_val_acc': 0.0}

def save_manifest(m):
    with open(MANIFEST_PATH, 'w') as f:
        json.dump(m, f, indent=2)

def ckpt_path(epoch):
    return os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:02d}.pt')

@torch.inference_mode()
def evaluate(loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return correct / total

manifest      = load_manifest()
done_epochs   = {c['epoch'] for c in manifest['checkpoints']}
best_val_acc  = manifest['best_val_acc']
criterion     = nn.CrossEntropyLoss()

# ── Determine resume state ───────────────────────────────────────────────────
# Stage A epochs are labelled 1-3, Stage B epochs 4-18 (3 + up to 15)
last_done = max(done_epochs) if done_epochs else 0
if last_done > 0:
    print(f'Resuming from epoch {last_done}. Loading weights...')
    model.load_state_dict(torch.load(ckpt_path(last_done), map_location=DEVICE))
    print(f'Best so far: epoch {manifest["best_epoch"]} @ val={best_val_acc:.4f}')

# ── Stage A ──────────────────────────────────────────────────────────────────
STAGE_A_EPOCHS = 3

if last_done < STAGE_A_EPOCHS:
    print('\n=== Stage A: backbone frozen, head only ===')
    for p in model.features.parameters():
        p.requires_grad = False
    opt = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    for epoch in range(last_done + 1, STAGE_A_EPOCHS + 1):
        t0 = time.time()
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            running += loss.item()

        val_acc = evaluate(val_loader)
        elapsed = time.time() - t0

        # Save checkpoint
        torch.save(model.state_dict(), ckpt_path(epoch))

        # Update manifest
        entry = {'epoch': epoch, 'stage': 'A', 'val_acc': round(val_acc, 4),
                 'loss': round(running/len(train_loader), 4), 'seconds': round(elapsed)}
        manifest['checkpoints'].append(entry)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            manifest['best_epoch']   = epoch
            manifest['best_val_acc'] = round(val_acc, 4)
            # Copy to best.pt
            import shutil
            shutil.copy2(ckpt_path(epoch), os.path.join(CHECKPOINT_DIR, 'best.pt'))
        save_manifest(manifest)

        print(f'  A epoch {epoch}/{STAGE_A_EPOCHS}  loss={running/len(train_loader):.4f}'
              f'  val={val_acc:.4f}  ({elapsed:.0f}s)  → saved epoch_{epoch:02d}.pt')
    last_done = STAGE_A_EPOCHS

# ── Stage B ──────────────────────────────────────────────────────────────────
STAGE_B_MAX  = 15
STAGE_B_START = STAGE_A_EPOCHS + 1
STAGE_B_END   = STAGE_A_EPOCHS + STAGE_B_MAX

if last_done < STAGE_B_END:
    print('\n=== Stage B: last conv block + head unfrozen ===')
    for p in model.parameters():
        p.requires_grad = False
    for p in model.features[-1].parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

    stage_b_epoch = max(last_done - STAGE_A_EPOCHS, 0)   # how many B epochs already done
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STAGE_B_MAX)
    # Fast-forward scheduler to match resume point
    for _ in range(stage_b_epoch):
        sched.step()

    patience   = 5
    no_improve = manifest.get('no_improve_b', 0)

    for epoch in range(last_done + 1, STAGE_B_END + 1):
        t0 = time.time()
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            running += loss.item()
        sched.step()

        val_acc = evaluate(val_loader)
        elapsed = time.time() - t0

        # Save checkpoint
        torch.save(model.state_dict(), ckpt_path(epoch))

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            no_improve   = 0
            manifest['best_epoch']   = epoch
            manifest['best_val_acc'] = round(val_acc, 4)
            import shutil
            shutil.copy2(ckpt_path(epoch), os.path.join(CHECKPOINT_DIR, 'best.pt'))
        else:
            no_improve += 1

        entry = {'epoch': epoch, 'stage': 'B', 'val_acc': round(val_acc, 4),
                 'loss': round(running/len(train_loader), 4), 'seconds': round(elapsed)}
        manifest['checkpoints'].append(entry)
        manifest['no_improve_b'] = no_improve
        save_manifest(manifest)

        star = ' ★ new best' if improved else ''
        print(f'  B epoch {epoch-STAGE_A_EPOCHS}/{STAGE_B_MAX}  (global {epoch})'
              f'  loss={running/len(train_loader):.4f}  val={val_acc:.4f}'
              f'  ({elapsed:.0f}s)  → epoch_{epoch:02d}.pt{star}')

        if no_improve >= patience:
            print(f'  Early stop: {patience} epochs without improvement.')
            break

print(f'\n✓ Training complete. Best: epoch {manifest["best_epoch"]} val={manifest["best_val_acc"]}')

## Step 6: View All Checkpoints
Shows every saved epoch and its val accuracy so you can pick one.

In [ ]:
import json
MANIFEST_PATH = os.path.join(CHECKPOINT_DIR, 'manifest.json')
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print(f'Best checkpoint: epoch {manifest["best_epoch"]}  val_acc={manifest["best_val_acc"]}\n')
print(f'{"Epoch":>6}  {"Stage":>5}  {"Val Acc":>8}  {"Loss":>7}  {"Time":>6}  File')
print('-' * 65)
for c in manifest['checkpoints']:
    star = ' ★' if c['epoch'] == manifest['best_epoch'] else ''
    print(f'{c["epoch"]:>6}  {c["stage"]:>5}  {c["val_acc"]:>8.4f}  '
          f'{c["loss"]:>7.4f}  {c["seconds"]:>5}s  epoch_{c["epoch"]:02d}.pt{star}')

## Step 7: Load & Test Any Checkpoint

Set `EPOCH_TO_TEST` to any epoch number from the table above.  
The cell loads that checkpoint, shows 10 random val images with predictions,
and saves it as `english_sign_classifier.pt` + `english_classifier_meta.json` to Drive
then auto-downloads both to your browser.

In [ ]:
# ── CONFIGURE THIS ────────────────────────────────────────────────────────────
EPOCH_TO_TEST = None   # set to e.g. 7, or None to use the best checkpoint automatically
# ─────────────────────────────────────────────────────────────────────────────

import json, random, shutil
import matplotlib.pyplot as plt
from PIL import Image

MANIFEST_PATH = os.path.join(CHECKPOINT_DIR, 'manifest.json')
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

epoch = EPOCH_TO_TEST if EPOCH_TO_TEST is not None else manifest['best_epoch']
pt    = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:02d}.pt')
assert os.path.exists(pt), f'Checkpoint not found: {pt}'

# Match the entry from manifest
entry = next((c for c in manifest['checkpoints'] if c['epoch'] == epoch), {})
print(f'Loading epoch {epoch}  (stage {entry.get("stage","?")}  val_acc={entry.get("val_acc","?")})\n')

model.load_state_dict(torch.load(pt, map_location=DEVICE))
model.eval()

# Evaluate on full val set
val_acc = evaluate(val_loader)
print(f'Val accuracy (full set): {val_acc:.4f}')

# Show 10 random val images
val_root = os.path.join(CROPPED_DIR, 'val')
samples  = []
for cls in CLASS_NAMES:
    d = os.path.join(val_root, cls)
    if os.path.isdir(d) and os.listdir(d):
        samples.append((cls, os.path.join(d, random.choice(os.listdir(d)))))
random.shuffle(samples)
samples = samples[:10]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (true_cls, img_path) in zip(axes.flatten(), samples):
    img_pil = Image.open(img_path).convert('RGB')
    tensor  = val_tf(img_pil).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        probs = torch.softmax(model(tensor), dim=1)[0]
    conf, idx = probs.max(0)
    pred = CLASS_NAMES[idx.item()]
    ax.imshow(img_pil)
    ax.set_title(f'True: {true_cls}\nPred: {pred} ({conf:.2f})',
                 color='green' if pred==true_cls else 'red', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

# Save as the production model
final_pt   = os.path.join(PROJECT_DIR, 'english_sign_classifier.pt')
final_meta = os.path.join(PROJECT_DIR, 'english_classifier_meta.json')
shutil.copy2(pt, final_pt)
with open(final_meta, 'w') as f:
    json.dump({
        'backbone': 'mobilenet_v3_small', 'input_size': 224,
        'mean': [0.485,0.456,0.406], 'std': [0.229,0.224,0.225],
        'classes': CLASS_NAMES, 'num_classes': len(CLASS_NAMES),
        'source_epoch': epoch, 'val_accuracy': round(val_acc, 4),
    }, f, indent=2)

print(f'\nSaved to Drive as english_sign_classifier.pt')

# Download to browser
from google.colab import files
files.download(final_pt)
files.download(final_meta)
print('Downloads started → drop both into your local models/ folder.')

## Step 8: Train Landmark MLP & Export

Extracts 63-float normalised MediaPipe landmark vectors from the already-cropped
training images, trains a tiny 63 → 256 → 128 → 29 MLP, and saves it alongside the CNN.

**Why this helps**: the CNN learns from pixels and struggles with letters whose hand
shapes look similar at low resolution (D/E, M/N/T, U/V).  The MLP learns from
explicit finger geometry — it directly encodes which fingers are extended and how far
apart they are, so it has complementary errors.  Averaging both probability vectors
(ensemble) reliably outperforms either model alone on hard look-alike classes.

- First run: ~5–10 min on T4 (landmark extraction + 40-epoch MLP training).
- Output: `english_landmark_mlp.pt` saved to Drive and auto-downloaded.
- Drop the file into your local `models/` folder — the app detects it automatically
  on the next startup and switches to ensemble mode (no code change needed).


In [ ]:
# ── Self-contained setup (safe to run after a runtime restart) ────────────────
import os, cv2, numpy as np, time, threading, concurrent.futures
import torch, torch.nn as nn

PROJECT_DIR = '/content/drive/MyDrive/ArSL_Project'
CROPPED_DIR = os.path.join(PROJECT_DIR, 'asl_cropped')
CLASS_NAMES = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','SPACE','DEL','NOTHING'
]

from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
except Exception:
    drive.mount('/content/drive')

# ── MediaPipe API detection ───────────────────────────────────────────────────
import mediapipe as mp
print(f"mediapipe {mp.__version__}")

_USE_LEGACY = False
try:
    _ = mp.solutions.hands.Hands
    _USE_LEGACY = True
    print("Using legacy mp.solutions API")
except AttributeError:
    print("solutions API absent — using MediaPipe Tasks API")
    import urllib.request
    _MODEL_PATH = "/tmp/hand_landmarker.task"
    if not os.path.exists(_MODEL_PATH):
        print("Downloading hand_landmarker.task (~8 MB)...")
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/"
            "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
            _MODEL_PATH,
        )
        print("Downloaded.")

# ── CONFIG ────────────────────────────────────────────────────────────────────
MLP_EPOCHS     = 40
MLP_BATCH_SIZE = 512
MLP_LR         = 1e-3
MLP_SAVE_PATH  = os.path.join(PROJECT_DIR, "english_landmark_mlp.pt")
NUM_WORKERS    = 8   # parallel threads for landmark extraction

_WRIST      = 0
_MIDDLE_MCP = 9

def _normalise(pts_21x3):
    pts   = pts_21x3.copy()
    pts  -= pts[_WRIST]
    scale = float(np.linalg.norm(pts[_MIDDLE_MCP])) + 1e-6
    pts  /= scale
    return pts.flatten()

# ── Per-API context factory ───────────────────────────────────────────────────
if _USE_LEGACY:
    def _make_ctx():
        return mp.solutions.hands.Hands(
            static_image_mode=True, max_num_hands=1,
            min_detection_confidence=0.3)

    def _run_ctx(img_bgr, ctx):
        rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        result = ctx.process(rgb)
        if not result.multi_hand_landmarks:
            return None
        lms = result.multi_hand_landmarks[0]
        pts = np.array([[l.x, l.y, l.z] for l in lms.landmark], dtype=np.float32)
        return _normalise(pts)
else:
    from mediapipe.tasks import python as _mp_tasks
    from mediapipe.tasks.python import vision as _mp_vision
    _lm_opts = _mp_vision.HandLandmarkerOptions(
        base_options=_mp_tasks.BaseOptions(model_asset_path=_MODEL_PATH),
        running_mode=_mp_vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.3,
        min_tracking_confidence=0.3,
    )

    def _make_ctx():
        return _mp_vision.HandLandmarker.create_from_options(_lm_opts)

    def _run_ctx(img_bgr, ctx):
        rgb    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = ctx.detect(mp_img)
        if not result.hand_landmarks:
            return None
        lms = result.hand_landmarks[0]
        pts = np.array([[l.x, l.y, l.z] for l in lms], dtype=np.float32)
        return _normalise(pts)

# ── Thread-local context pool ─────────────────────────────────────────────────
_tls       = threading.local()
_all_ctxs  = []
_ctx_lock  = threading.Lock()

def _tls_ctx():
    """One MediaPipe context per worker thread, created lazily."""
    if not hasattr(_tls, 'ctx'):
        _tls.ctx = _make_ctx()
        with _ctx_lock:
            _all_ctxs.append(_tls.ctx)
    return _tls.ctx

def _process_one(img_path_cls):
    img_path, cls_idx = img_path_cls
    img = cv2.imread(img_path)
    if img is None:
        return None
    feat = _run_ctx(img, _tls_ctx())
    return (feat, cls_idx) if feat is not None else None

# ── Step 8a: Parallel landmark extraction ────────────────────────────────────
FEAT_CACHE = os.path.join(PROJECT_DIR, "landmark_features.npz")

if os.path.exists(FEAT_CACHE):
    print("Loading cached landmark features...")
    data = np.load(FEAT_CACHE)
    X_train, y_train = data["X_train"], data["y_train"]
    X_val,   y_val   = data["X_val"],   data["y_val"]
    print(f"Train: {len(X_train)}, Val: {len(X_val)}")
else:
    def collect_split(split):
        split_dir = os.path.join(CROPPED_DIR, split)
        all_tasks = []
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            cls_dir = os.path.join(split_dir, cls_name)
            if not os.path.isdir(cls_dir):
                print(f"  WARNING: {cls_dir} not found"); continue
            imgs = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
                    if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            all_tasks.extend((p, cls_idx) for p in imgs)

        X_list, y_list, ok, skip = [], [], 0, 0
        t0 = time.time()
        with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as exe:
            for i, res in enumerate(exe.map(_process_one, all_tasks, chunksize=256)):
                if res is None:
                    skip += 1
                else:
                    X_list.append(res[0])
                    y_list.append(res[1])
                    ok += 1
                if (i + 1) % 10000 == 0:
                    elapsed = time.time() - t0
                    rate    = (i + 1) / elapsed
                    remain  = (len(all_tasks) - i - 1) / rate
                    print(f"  {split}: {i+1}/{len(all_tasks)}  "
                          f"{rate:.0f} img/s  ~{remain:.0f}s left")

        # Close all thread-local contexts created for this split
        for ctx in _all_ctxs:
            try: ctx.close()
            except Exception: pass
        _all_ctxs.clear()

        print(f"  {split} done: {ok} ok, {skip} skipped  "
              f"({time.time()-t0:.0f}s total)")
        return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)

    print(f"Extracting landmarks with {NUM_WORKERS} parallel workers...")
    X_train, y_train = collect_split("train")
    X_val,   y_val   = collect_split("val")
    np.savez_compressed(FEAT_CACHE, X_train=X_train, y_train=y_train,
                                     X_val=X_val,   y_val=y_val)
    print("Features cached to Drive as landmark_features.npz")

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# ── Step 8b: Train the landmark MLP ──────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {DEVICE}")

class LandmarkMLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.net(x)

mlp       = LandmarkMLP(len(CLASS_NAMES)).to(DEVICE)
opt       = torch.optim.Adam(mlp.parameters(), lr=MLP_LR)
criterion = nn.CrossEntropyLoss()

Xt = torch.from_numpy(X_train)
yt = torch.from_numpy(y_train)
Xv = torch.from_numpy(X_val).to(DEVICE)
yv = torch.from_numpy(y_val).to(DEVICE)

n            = len(Xt)
best_val_acc = 0.0
best_state   = None

print(f"Training MLP for {MLP_EPOCHS} epochs on {n} samples...")
for epoch in range(1, MLP_EPOCHS + 1):
    mlp.train()
    perm       = torch.randperm(n)
    Xt_shuf    = Xt[perm];  yt_shuf = yt[perm]
    total_loss = 0.0
    for i in range(0, n, MLP_BATCH_SIZE):
        xb = Xt_shuf[i:i+MLP_BATCH_SIZE].to(DEVICE)
        yb = yt_shuf[i:i+MLP_BATCH_SIZE].to(DEVICE)
        opt.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward();  opt.step()
        total_loss += loss.item() * len(xb)
    mlp.eval()
    with torch.inference_mode():
        val_acc = (mlp(Xv).argmax(1) == yv).float().mean().item()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
    if epoch % 5 == 0 or epoch == 1:
        print(f"  epoch {epoch:3d}/{MLP_EPOCHS}  "
              f"loss={total_loss/n:.4f}  val_acc={val_acc:.4f}")

print(f"\nBest MLP val accuracy: {best_val_acc:.4f}")

# ── Step 8c: Save + download ──────────────────────────────────────────────────
torch.save({"state_dict": best_state, "classes": CLASS_NAMES,
            "val_accuracy": round(best_val_acc, 4)}, MLP_SAVE_PATH)
print(f"Saved to Drive: {MLP_SAVE_PATH}")

from google.colab import files
files.download(MLP_SAVE_PATH)
print("Download started → drop english_landmark_mlp.pt into your local models/ folder.")
print("The app detects it on next startup and switches to ensemble mode.")
